In [1]:
import cme.decision_models.confidence_accumulation as ca
import numpy as np
import jax
import numpyro as npy
import numpyro.distributions as dist
import pandas as pd
import seaborn as sns
import scipy.stats as stats


In [2]:
rng = jax.random.key(1)
rng

Array((), dtype=key<fry>) overlaying:
[0 1]

In [3]:
dist.Beta(2,2).sample(rng, sample_shape=(5,10))

Array([[0.29949295, 0.73421339, 0.42720974, 0.43203879, 0.88556774,
        0.28975586, 0.2804195 , 0.52809878, 0.71082405, 0.72814774],
       [0.18972341, 0.60960151, 0.14057264, 0.69799797, 0.39955407,
        0.81406206, 0.5340404 , 0.39418802, 0.26261299, 0.76119076],
       [0.16773461, 0.48625116, 0.83507075, 0.80796672, 0.80803134,
        0.86572982, 0.80349066, 0.43938511, 0.55053788, 0.57353783],
       [0.79444017, 0.13349048, 0.56735756, 0.72874945, 0.38638623,
        0.64232726, 0.41170649, 0.50453663, 0.6687633 , 0.79843571],
       [0.40804458, 0.37166234, 0.59654398, 0.23979649, 0.37745059,
        0.57452887, 0.17705557, 0.83192597, 0.94653671, 0.8171046 ]],      dtype=float64)

In [4]:
n_states, start_width, threshold, delta, measurement_prob, delta, mu, sigma, I, J = 11, 3, 2, 1, 0.8, 0.01, np.asarray([[2]]), np.asarray([[1]]), 10, 5
model_type = "Quantum"


In [5]:
Mc, Mw, Mn = ca._get_measurement_matrix(n_states, threshold, prob=measurement_prob, model_type = model_type)
Mc

Array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.

In [6]:
intensity_matrix = ca.get_intensity_matrix(n_states, mu, sigma, model_type=model_type)
phi_0 = ca._get_initial_state(n_states, start_width,model_type=model_type, prior_type="Uniform")
intensity_matrix, phi_0

(Array([[[[0.+10.j, 0. -1.j, 0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j,
           0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j],
          [0. -1.j, 0. +8.j, 0. -1.j, 0. -0.j, 0. -0.j, 0. -0.j,
           0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j],
          [0. -0.j, 0. -1.j, 0. +6.j, 0. -1.j, 0. -0.j, 0. -0.j,
           0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j],
          [0. -0.j, 0. -0.j, 0. -1.j, 0. +4.j, 0. -1.j, 0. -0.j,
           0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j],
          [0. -0.j, 0. -0.j, 0. -0.j, 0. -1.j, 0. +2.j, 0. -1.j,
           0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j],
          [0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j, 0. -1.j, 0. -0.j,
           0. -1.j, 0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j],
          [0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j, 0. -1.j,
           0. -2.j, 0. -1.j, 0. -0.j, 0. -0.j, 0. -0.j],
          [0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j, 0. -0.j,
           0. -1.j, 0. -4.j, 0. -1.j, 0. -0.j, 0. -0.j],
          [0. -0.j, 0. -

In [7]:
t = np.asarray([[4.5]])
phi_t = ca.perform_state_transition(intensity_matrix=intensity_matrix, RT_s=t, RA_s = None, delta=delta, 
Mc = Mc, Mn = Mn, Mw = Mw, phi_0=phi_0, 
transition_type="TIMESTEP", likelihood_type="SINGLE")
phi_t

Array([[[[ 1.02690886e-05+3.57417673e-05j],
         [-2.71917265e-03+9.20448442e-04j],
         [-5.51158770e-02-1.50609098e-01j],
         [ 1.23627618e-01-2.59195312e-02j],
         [-2.03349178e-01+1.38314030e-01j],
         [ 2.71018709e-01-9.83579751e-02j],
         [-3.08093420e-01+2.56846646e-02j],
         [ 2.23951193e-01+3.03546640e-01j],
         [-3.98906218e-01+1.74008794e-01j],
         [ 3.39192380e-03+7.19495062e-03j],
         [ 9.24970194e-05-4.96001557e-05j]]]], dtype=complex128)

In [8]:
S = np.where(dist.MultinomialProbs(phi_t.squeeze()).sample(rng))[0][0]
S

8

rt = []
ra = []
if S < start_width:
    rt.append(t)
    ra.append(0)
elif S > n_states - start_width:
    rt.append(t)
    ra.append(1)

rt, ra

In [9]:
stats.uniform(0,1).rvs()

0.39287559110248027

In [10]:
#stats.multinomial(p=phi_t)

In [11]:
rng = jax.random.key(1)
dist.Uniform().sample(rng), jax.random.uniform(rng)

(Array(0.45375874, dtype=float64), Array(0.45375874, dtype=float64))

In [12]:
def sample_state_1(phi):
    #return np.where(dist.MultinomialProbs(phi_t).sample(rng))[0][0] + 1
    return np.where(stats.multinomial(phi_t).rvs()) + 1

def sample_state(phi):
    phi_cumsum = np.cumsum(phi)
    ran_prob = stats.uniform(0,1).rvs() #dist.Uniform().sample(rng)
    #print(ran_prob)
    S = np.argmax(ran_prob < phi_cumsum)
    return S + 1 # due to 0-based indexing

def random_walk(intensity_matrix, phi_0, delta, T_max, model_type):
    ra = -1
    rt = -1
    phi_arr = []
    for t in np.arange(delta, T_max, delta):
        #print(t)
        phi_t = ca.perform_state_transition(intensity_matrix=intensity_matrix, RT_s=np.asarray([[t]]), RA_s = None, delta=delta, 
                                            Mc = Mc, Mn = Mn, Mw = Mw, phi_0=phi_0, 
                                            transition_type="TIMESTEP", likelihood_type="SINGLE")
        if model_type == model_type:
            phi_t = np.abs(phi_t)**2
        phi_t = phi_t.squeeze()/phi_t.sum()
        S = sample_state(phi_t) # adding 1 because where returns 0-based indexing
        #print(S, phi_t.sum())
        #states.append(S)
        phi_arr.append(phi_t)
        if S < threshold:
            ra=0
            rt=t
            break
        elif S > n_states - threshold:
            ra=1
            rt=t
            #print(phi_t, S, t)
            break
    return rt, ra, S, phi_arr

In [13]:
rt_s = []
ra_s = []
for _ in range(1):
    t, ra, S, phi_arr = random_walk(intensity_matrix, phi_0, delta, 100, model_type)
    rt_s.append(t)
    ra_s.append(ra)
rt_s

In [18]:
def get_samples():
    df_x = []
    mus = stats.norm().rvs(10)
    for mu in mus: 
        intensity_matrix = ca.get_intensity_matrix(n_states, np.asarray([[mu]]), sigma, model_type=model_type)
        phi_0 = ca._get_initial_state(n_states, start_width,model_type=model_type, prior_type="Uniform")
        for _ in range(1):
            t, ra, S, phi_arr = random_walk(intensity_matrix, phi_0, delta, 100, model_type)
            rt_s.append(t)
            ra_s.append(ra)
        df_x.append(pd.DataFrame({"rt":rt_s, "ra":ra_s, "mu":mu}))    
    df_x = pd.concat(df_x)
    return df_x
df_x = get_samples()
df_x

JIT session error: Cannot allocate memory


XlaRuntimeError: INVALID_ARGUMENT: Symbol main.22 not found.

In [ ]:
sns.kdeplot(rt_s)